In [1]:
import h5py
import numpy as np
import pandas as pd

In [3]:
with h5py.File('../data/2026-04-07_13-08-05_particles.h5', 'r') as f:
    def print_structure(name, obj):
        print(name, ':', type(obj).__name__, end='')
        if hasattr(obj, 'shape'):
            print(f'  shape={obj.shape}  dtype={obj.dtype}', end='')
        print()
    f.visititems(print_structure)

meta : Group
meta/numParticles : Dataset  shape=(1,)  dtype=int32
meta/numSteps : Dataset  shape=(1,)  dtype=int32
meta/saveEvery : Dataset  shape=(1,)  dtype=int32
params : Group
params/diffusion : Dataset  shape=(1,)  dtype=float64
params/dt : Dataset  shape=(1,)  dtype=float64
params/manifold_type : Dataset  shape=(1,)  dtype=int8
params/mobility : Dataset  shape=(1,)  dtype=float64
params/potential_strength : Dataset  shape=(1,)  dtype=float64
params/radius : Dataset  shape=(1,)  dtype=float64
params/seed : Dataset  shape=(1,)  dtype=uint32
params/v : Dataset  shape=(1,)  dtype=float64
state : Group
state/q1 : Dataset  shape=(500, 50)  dtype=float64
state/q2 : Dataset  shape=(500, 50)  dtype=float64
state/theta : Dataset  shape=(500, 50)  dtype=float64


In [10]:
with h5py.File('../data/2026-04-07_13-08-05_particles.h5', 'r') as f:
    
    # --- Metadata ---
    num_particles = np.asarray(f['meta/numParticles'])[0]
    num_steps     = np.asarray(f['meta/numSteps'])[0] 
    save_every    =  np.asarray(f['meta/saveEvery'])[0]

    # --- Simulation parameters (scalar, just read directly) ---
    params = {key: f[f'params/{key}'][0] for key in f['params'].keys()}  # type: ignore

    # --- State arrays: shape (num_steps, num_particles) ---
    q1    = np.asarray(f['state/q1'])[:]    
    q2    = np.asarray(f['state/q2'])[:]    # (500, 50) 
    theta = np.asarray(f['state/theta'])[:] # (500, 50)  

num_saved, num_particles = q1.shape  # 500, 50  — ground truth from the data

time_index = np.arange(num_saved) * save_every * dt

steps     = np.repeat(np.arange(num_saved), num_particles)
times     = np.repeat(time_index, num_particles)
particles = np.tile(np.arange(num_particles), num_saved)

df = pd.DataFrame({
    'step':     steps,
    'time':     times,
    'particle': particles,
    'q1':       q1.ravel(),
    'q2':       q2.ravel(),
    'theta':    theta.ravel(),
})

print(df.shape)   # (25000, 6)
print(df.head())


(25000, 6)
   step  time  particle        q1        q2     theta
0     0   0.0         0  1.877172  3.107600  2.783545
1     0   0.0         1  5.227053  3.665119  0.158159
2     0   0.0         2  4.456085  1.668601  1.656266
3     0   0.0         3  0.944852  4.296558  5.130861
4     0   0.0         4  2.111600  5.597165  1.244836
